In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 4, Finished, Available, Finished, False)

###### **- <mark>**Silver layer implemention Plan**</mark>:

###### 1. Trim whitespace
###### 2. Lowercase normalization
###### 3. Regex cleaning
###### 4. Phone number masking
###### 5. Email domain extraction
###### 6. Type casting
###### 7. Data drift detection
###### 8. Null handling
###### 9. Deduplication
###### 10. Column standardization
###### 11. Data quality checks

**<mark>Step 0 — Load Bronze Table (Preparation Step)</mark>**

In [3]:
customers_df = spark.table("bronze_customers")

customers_df.printSchema()
customers_df.show(5)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 5, Finished, Available, Finished, False)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- country: string (nullable = true)
 |-- address: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- loyalty_points: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- notes: string (nullable = true)

+-----------+------------------+--------------------+--------------+-------+--------------------+-----------+--------------+----------------+--------------------+--------------------+
|customer_id|     customer_name|               email|  phone_number|country|             address|signup_date|loyalty_points|customer_segment|          promo_code|               notes|
+-----------+------------------+--------------------+--------------+-------+--------------------+-----------+--------------+----------------+--------------------+-

<mark>**Transformation 1 — Trim Whitespace** </mark>
 Why This Is Needed ?

- Real datasets often contain leading or trailing spaces.

In [4]:
from pyspark.sql.functions import trim

customers_df = customers_df.withColumn(
    "customer_name",
    trim("customer_name")
)

customers_df = customers_df.withColumn(
    "email",
    trim("email")
)

customers_df = customers_df.withColumn(
    "country",
    trim("country")
)

customers_df = customers_df.withColumn(
    "customer_segment",
    trim("customer_segment")
)

customers_df = customers_df.withColumn(
    "promo_code",
    trim("promo_code")
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 6, Finished, Available, Finished, False)

<mark>**Validate Transformation - white space Trim**</mark>

In [5]:
customers_df.select(
    "customer_name",
    "email",
    "country"
).show(5, truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 7, Finished, Available, Finished, False)

+------------------+--------------------------+-------+
|customer_name     |email                     |country|
+------------------+--------------------------+-------+
|Jessica Patel     |amandagarrett@example.com |IN     |
|Christopher Martin|juarezstephen@example.net |IN     |
|Andrew Lee        |ryangonzalez@example.org  |AU     |
|Dawn Smith        |ngross@example.com        |IN     |
|Denise Jackson    |williamsandrew@example.net|IN     |
+------------------+--------------------------+-------+
only showing top 5 rows



 <mark><mark></mark>**Transformation 2 — Lowercase Normalization**</mark>
- Why This Is Important ?

-  Real data often contains inconsistent casing.

In [6]:
from pyspark.sql.functions import lower

customers_df = customers_df.withColumn(
    "email",
    lower("email")
)

customers_df = customers_df.withColumn(
    "country",
    lower("country")
)

customers_df = customers_df.withColumn(
    "customer_segment",
    lower("customer_segment")
)

customers_df = customers_df.withColumn(
    "promo_code",
    lower("promo_code")
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 8, Finished, Available, Finished, False)

<mark>**Validate the Transformation - lower case normalization**</mark>

In [7]:
customers_df.select(
    "email",
    "country",
    "customer_segment",
    "promo_code"
).show(5, truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 9, Finished, Available, Finished, False)

+--------------------------+-------+----------------+---------------------+
|email                     |country|customer_segment|promo_code           |
+--------------------------+-------+----------------+---------------------+
|amandagarrett@example.com |in     |vip             |sale-2024-summer     |
|juarezstephen@example.net |in     |regular         |sale-2024-blackfriday|
|ryangonzalez@example.org  |au     |regular         |sale-2024-newyear    |
|ngross@example.com        |in     |regular         |sale-2024-diwali     |
|williamsandrew@example.net|in     |vip             |sale-2024-diwali     |
+--------------------------+-------+----------------+---------------------+
only showing top 5 rows



###### <mark>**Transformation 3 — Regexp Cleaning**</mark>
**Goal of This Transformation is to - <mark>
###### Convert phone numbers like +98-8604055752  ---->>>    988604055752 using --> <mark>**regexp_replace()**</mark>**
</mark>
**

In [8]:
from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "phone_clean",
    regexp_replace("phone_number", "[^0-9]", "")
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 10, Finished, Available, Finished, False)

<mark>**Validate the Transformation regexp_replace**</mark>

In [9]:
customers_df.select(
    "phone_number",
    "phone_clean"
).show(5, truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 11, Finished, Available, Finished, False)

+--------------+------------+
|phone_number  |phone_clean |
+--------------+------------+
|+98-8604055752|988604055752|
|+66-8676977143|668676977143|
|+56-8551098126|568551098126|
|+81-8087262945|818087262945|
|+42-9839030853|429839030853|
+--------------+------------+
only showing top 5 rows



###### <mark>****Transformation 4 — Email Domain Extraction using -  regexp_extract()** **</mark>

In [10]:
from pyspark.sql.functions import regexp_extract

customers_df = customers_df.withColumn(
    "email_domain",
    regexp_extract("email", "@(.+)", 1)
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 12, Finished, Available, Finished, False)

<mark>**Validate the Transformation - Email Domain Extraction**</mark>

In [11]:
customers_df.select(
    "email",
    "email_domain"
).show(5, truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 13, Finished, Available, Finished, False)

+--------------------------+------------+
|email                     |email_domain|
+--------------------------+------------+
|amandagarrett@example.com |example.com |
|juarezstephen@example.net |example.net |
|ryangonzalez@example.org  |example.org |
|ngross@example.com        |example.com |
|williamsandrew@example.net|example.net |
+--------------------------+------------+
only showing top 5 rows



###### <mark>**Transformation 5 — Phone Number Masking**</mark>

In [12]:
from pyspark.sql.functions import expr

customers_df = customers_df.withColumn(
    "phone_masked",
    expr("""
        concat(
            substr(phone_clean, 1, 2),
            repeat('*', length(phone_clean) - 4),
            right(phone_clean, 2)
        )
    """)
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 14, Finished, Available, Finished, False)

<mark>**Validate Transformation 5 — Phone Number Masking**</mark>

In [13]:
customers_df.select(
    "phone_clean",
    "phone_masked"
).show(10, truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 15, Finished, Available, Finished, False)

+------------+------------+
|phone_clean |phone_masked|
+------------+------------+
|988604055752|98********52|
|668676977143|66********43|
|568551098126|56********26|
|818087262945|81********45|
|429839030853|42********53|
|97913365780 |97*******80 |
|768708560006|76********06|
|759982314912|75********12|
|588239565083|58********83|
|659745555896|65********96|
+------------+------------+
only showing top 10 rows



In [14]:
from pyspark.sql.functions import length

customers_df.select(
    "phone_clean",
    "phone_masked",
    length("phone_clean").alias("original_length"),
    length("phone_masked").alias("masked_length")
).show(10, False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 16, Finished, Available, Finished, False)

+------------+------------+---------------+-------------+
|phone_clean |phone_masked|original_length|masked_length|
+------------+------------+---------------+-------------+
|988604055752|98********52|12             |12           |
|668676977143|66********43|12             |12           |
|568551098126|56********26|12             |12           |
|818087262945|81********45|12             |12           |
|429839030853|42********53|12             |12           |
|97913365780 |97*******80 |11             |11           |
|768708560006|76********06|12             |12           |
|759982314912|75********12|12             |12           |
|588239565083|58********83|12             |12           |
|659745555896|65********96|12             |12           |
+------------+------------+---------------+-------------+
only showing top 10 rows



###### <mark>**Transformation 6 — Type Casting**</mark>

In [15]:
from pyspark.sql.functions import col

customers_df = customers_df.withColumn(
    "customer_id",
    col("customer_id").cast("int")
)

customers_df = customers_df.withColumn(
    "loyalty_points",
    col("loyalty_points").cast("int")
)

customers_df = customers_df.withColumn(
    "signup_date",
    col("signup_date").cast("date")
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 17, Finished, Available, Finished, False)

###### **<mark>Validate Schema</mark><mark></mark>**

In [16]:
customers_df.printSchema()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 18, Finished, Available, Finished, False)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- country: string (nullable = true)
 |-- address: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- loyalty_points: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- phone_clean: string (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- phone_masked: string (nullable = true)



###### <mark>**Transformation 7 — Null Handling**</mark>

In [17]:
from pyspark.sql.functions import coalesce, lit

customers_df = customers_df.withColumn(
    "customer_segment",
    coalesce("customer_segment", lit("unknown"))
)

customers_df = customers_df.withColumn(
    "promo_code",
    coalesce("promo_code", lit("no_promo"))
)

customers_df = customers_df.withColumn(
    "notes",
    coalesce("notes", lit("no_notes"))
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 19, Finished, Available, Finished, False)

###### <mark>**Validate Transformation - Null Handling with coalesce() lit()**</mark>

In [18]:
customers_df.select(
    "customer_segment",
    "promo_code",
    "notes"
).show(5, truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 20, Finished, Available, Finished, False)

+----------------+---------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_segment|promo_code           |notes                                                                                                                                                                                                  |
+----------------+---------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|vip             |sale-2024-summer     |Quickly new forget name see away far.\nTrial meeting simply not. Possible over man draw dark get somebody.\nFund stage effort. Wall least young return phone interest.                                 |
|regular         |sale-2024-blackfri

###### <mark>**Transformation 8 — Data Drift Detection**</mark>
<mark>**Why Data Drift Detection Is Important**</mark>

- Without drift detection:

- dashboards may break
 
- ML models may fail
 
- pipelines may silently produce wrong results
 
- Real pipelines often include data quality monitoring.

###### **<mark>Step 1 — Check Distinct Values</mark>**
**Expected values:**
vip
regular
unknown

**If something else appears → drift.**

In [19]:
customers_df.select("customer_segment").distinct().show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 21, Finished, Available, Finished, False)

+----------------+
|customer_segment|
+----------------+
|             vip|
|         regular|
|         premium|
+----------------+



###### **<mark>Step 2 — Check Country Values</mark>**
**If you see something like :-** 
**india**
**usa**

**that indicates source drift.**

In [20]:
customers_df.select("country").distinct().show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 22, Finished, Available, Finished, False)

+-------+
|country|
+-------+
|     us|
|     in|
|     au|
|     uk|
|     de|
+-------+



###### <mark>**Step 3 — Detect Invalid Values**</mark>
**Expected values :-**
<mark>**vip**
**regular**
**unknown**</mark>

**<mark>If something else appears → drift.</mark>**

In [21]:
customers_df.filter(
    ~customers_df.customer_segment.isin("vip", "regular", "unknown")
).show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 23, Finished, Available, Finished, False)

+-----------+----------------+--------------------+--------------+-------+--------------------+-----------+--------------+----------------+--------------------+--------------------+------------+------------+------------+
|customer_id|   customer_name|               email|  phone_number|country|             address|signup_date|loyalty_points|customer_segment|          promo_code|               notes| phone_clean|email_domain|phone_masked|
+-----------+----------------+--------------------+--------------+-------+--------------------+-----------+--------------+----------------+--------------------+--------------------+------------+------------+------------+
|          6|Gloria Rodriguez|zalvarez@example.org| +9-7913365780|     in|81905 Ortega Squa...| 2021-10-07|          3748|         premium|sale-2024-blackfr...|From available be...| 97913365780| example.org| 97*******80|
|          9| Carlos Arellano|  lori42@example.org|+58-8239565083|     in|579 Amanda Isle S...| 2021-07-15|         

###### <mark>**Transformation 9 — Deduplication**</mark>
**Why Deduplication Is Important**

**<mark>In real systems, duplicates occur due to:</mark>**

**multiple ingestion pipelines,**
**system retries,**
**CDC updates,**
**data synchronization issues**

**If duplicates remain:**

- analytics results become incorrect

- joins multiply records

- customer counts become inflated

In [22]:
customers_df = customers_df.dropDuplicates(["customer_id"])

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 24, Finished, Available, Finished, False)

###### <mark>**Validation of Deduplication**</mark>

In [23]:
customers_df.groupBy("customer_id") \
            .count() \
            .filter("count > 1") \
            .show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 25, Finished, Available, Finished, False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



###### <mark>**Transformation 10 — Column Standardization**</mark>

**Why Column Standardization Is Important**

**In many real datasets column names come from different systems and may look like:**

<mark>**customer_name,
cust_name,
customerName,
CUST_NAME**</mark>

**Without standardization:**

- **pipelines become inconsistent**

- **joins become confusing**

- **semantic models become messy**

In [24]:
customers_df = customers_df.withColumnRenamed(
    "customer_name",
    "customer_full_name"
)

customers_df = customers_df.withColumnRenamed(
    "phone_number",
    "phone_raw"
)

customers_df = customers_df.withColumnRenamed(
    "phone_clean",
    "phone_cleaned"
)

customers_df = customers_df.withColumnRenamed(
    "signup_date",
    "customer_signup_date"
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 26, Finished, Available, Finished, False)

###### <mark>**Validate Column Names**</mark>

In [25]:
customers_df.printSchema()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 27, Finished, Available, Finished, False)

root
 |-- customer_id: integer (nullable = true)
 |-- customer_full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_raw: string (nullable = true)
 |-- country: string (nullable = true)
 |-- address: string (nullable = true)
 |-- customer_signup_date: date (nullable = true)
 |-- loyalty_points: integer (nullable = true)
 |-- customer_segment: string (nullable = false)
 |-- promo_code: string (nullable = false)
 |-- notes: string (nullable = false)
 |-- phone_cleaned: string (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- phone_masked: string (nullable = true)



###### **<mark>Transformation 11 — Data Quality Checks</mark>**
**<mark>Why Data Quality Checks Are Important</mark>**

**In production pipelines we always validate data before moving to the next layer.**

**Typical validations include:**

- **Row count validation
- Null value checks
- Duplicate checks
- Unexpected value checks**

<mark>**These checks help detect:**</mark>

- **pipeline failures
- data corruption
- unexpected source changes**

###### <mark>**Step 1 — Row Count Validation**</mark>

In [26]:
customers_df.count()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 28, Finished, Available, Finished, False)

1000000

###### <mark>**Step 2 — Null Check on Critical Columns**</mark>


In [27]:
customers_df.filter("customer_id IS NULL").count()
customers_df.filter("email IS NULL").count()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 29, Finished, Available, Finished, False)

0

###### <mark>**Step 3 — Duplicate Check**</mark>
**Even though we removed duplicates earlier, we verify again.**

In [28]:
customers_df.groupBy("customer_id") \
            .count() \
            .filter("count > 1") \
            .show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 30, Finished, Available, Finished, False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



###### **<mark>Step 4 — Unexpected Value Check</mark>**
**Verify allowed values**.
- **vip
- regular
- unknown**

In [29]:
customers_df.select("customer_segment").distinct().show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 31, Finished, Available, Finished, False)

+----------------+
|customer_segment|
+----------------+
|             vip|
|         regular|
|         premium|
+----------------+



###### <mark>**After Validation - Write to Siver Table**</mark>

In [30]:
customers_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_customers")

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 32, Finished, Available, Finished, False)

###### **<mark>Next Phase — Phase 5</mark>**
**Multi-Table Transformations (Joins)**
**We will start combining below datasets:**

- customers
- orders
- order_items
- products
- payments

<mark>**Objective**</mark>

**Create enriched Silver tables that combine multiple datasets.**

###### **<mark>This join demonstrates:</mark>**

<mark>**dimension → fact relationship**</mark>

In [31]:
orders_df = spark.table("bronze_orders")

orders_df.printSchema()
orders_df.show(5)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 33, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_value: double (nullable = true)

+--------+-----------+----------+------------+-----------+
|order_id|customer_id|order_date|order_status|order_value|
+--------+-----------+----------+------------+-----------+
| 5200001|     672248|2025-12-11|      PLACED|    1864.77|
| 5200002|     527293|2025-12-20|   CANCELLED|     452.97|
| 5200003|     611730|2025-05-29|   DELIVERED|    1062.59|
| 5200004|     569894|2024-06-23|      PLACED|     170.17|
| 5200005|     263355|2025-01-04|     SHIPPED|     200.55|
+--------+-----------+----------+------------+-----------+
only showing top 5 rows



**Now we will create the first enriched dataset.**

In [32]:
customers_df = spark.table("silver_customers")

orders_df = spark.table("bronze_orders")

customer_orders_df = orders_df.join(
    customers_df,
    on="customer_id",
    how="inner")

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 34, Finished, Available, Finished, False)

<mark>**Join Validation**</mark>

In [33]:
customer_orders_df.show(3)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 35, Finished, Available, Finished, False)

+-----------+--------+----------+------------+-----------+------------------+-------------------+--------------+-------+--------------------+--------------------+--------------+----------------+-----------------+--------------------+-------------+------------+------------+
|customer_id|order_id|order_date|order_status|order_value|customer_full_name|              email|     phone_raw|country|             address|customer_signup_date|loyalty_points|customer_segment|       promo_code|               notes|phone_cleaned|email_domain|phone_masked|
+-----------+--------+----------+------------+-----------+------------------+-------------------+--------------+-------+--------------------+--------------------+--------------+----------------+-----------------+--------------------+-------------+------------+------------+
|          7| 3489530|2024-04-17|   CANCELLED|     522.63|        Maria Berg|fwalton@example.org|+76-8708560006|     in|8143 Lucas Forge ...|          2024-09-04|         15403| 

**<mark>Plan inspection</mark>**

In [34]:
customer_orders_df.explain()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 36, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#4412L, order_id#4411L, order_date#4413, order_status#4414, order_value#4415, customer_full_name#4384, email#4385, phone_raw#4386, country#4387, address#4388, customer_signup_date#4389, loyalty_points#4390, customer_segment#4391, promo_code#4392, notes#4393, phone_cleaned#4394, email_domain#4395, phone_masked#4396]
   +- SortMergeJoin [customer_id#4412L], [cast(customer_id#4383 as bigint)], Inner
      :- Sort [customer_id#4412L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(customer_id#4412L, 200), ENSURE_REQUIREMENTS, [plan_id=2411]
      :     +- FileScan parquet spark_catalog.chimcobldhq2ap9dcdnmqrb5e9hmabb6c5h74qb35lo74rraclhn89bcd1fmaorfdlmmasj3clfmqpb4c5m6oqbfdoim8ojf.bronze_orders[order_id#4411L,customer_id#4412L,order_date#4413,order_status#4414,order_value#4415] Batched: true, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://eaf0c26f-0ff8-484b-90

###### **<mark>This step will combine: orders  +  order_items</mark>**

###### **<mark>This is a Fact → Fact relationship, which is very common in e-commerce pipelines.</mark>**

**Step 1 — Inspect order_items Table**

In [35]:
order_items_df = spark.table("bronze_order_items")
order_items_df.printSchema()
order_items_df.show(3)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 37, Finished, Available, Finished, False)

root
 |-- item_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- item_price: double (nullable = true)

+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|item_price|
+-------+--------+----------+--------+----------+
|9200047| 3066683|     25709|       1|    233.04|
|9200048| 3066683|        44|       4|    450.42|
|9200049| 3066683|     18478|       4|      31.3|
+-------+--------+----------+--------+----------+
only showing top 3 rows



**Step 2 — Join orders with order_items**

In [36]:
orders_df = spark.table("bronze_orders")
order_items_df = spark.table("bronze_order_items")
orders_items_df = (
    orders_df
    .join(
        order_items_df,
        on="order_id",
        how="inner"
    )
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 38, Finished, Available, Finished, False)

**<mark>Validate Join</mark>**

In [37]:
orders_items_df.show(5)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 39, Finished, Available, Finished, False)

+--------+-----------+----------+------------+-----------+-------+----------+--------+----------+
|order_id|customer_id|order_date|order_status|order_value|item_id|product_id|quantity|item_price|
+--------+-----------+----------+------------+-----------+-------+----------+--------+----------+
|       7|     484757|2026-02-20|   CANCELLED|    1960.62|     19|     40914|       3|    420.47|
|       7|     484757|2026-02-20|   CANCELLED|    1960.62|     20|      4575|       2|    434.22|
|       7|     484757|2026-02-20|   CANCELLED|    1960.62|     21|     22345|       1|    374.77|
|      19|     649848|2024-05-26|      PLACED|    1051.56|     55|     41616|       5|     22.78|
|      19|     649848|2024-05-26|      PLACED|    1051.56|     56|     26565|       2|    476.71|
+--------+-----------+----------+------------+-----------+-------+----------+--------+----------+
only showing top 5 rows



###### **<mark>Step 4 — Check Row Count Increase</mark>**
**<mark>Because one order may contain multiple items.</mark>**

In [38]:
orders_df.count()
order_items_df.count()
orders_items_df.count()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 40, Finished, Available, Finished, False)

24000000

<mark>**We now join:**

**<mark>orders_items_df</mark>**
        **JOIN** 
**<mark>products</mark>**</mark>

In [39]:
products_df = spark.table("bronze_products")

orders_items_products_df = (
    orders_items_df
    .join(
        products_df,
        on="product_id",
        how="inner"
    )
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 41, Finished, Available, Finished, False)

###### **<mark>Validate Join</mark>**

In [40]:
orders_items_products_df.show(3)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 42, Finished, Available, Finished, False)

+----------+--------+-----------+----------+------------+-----------+-------+--------+----------+------------+-----------+------+------+--------------------+
|product_id|order_id|customer_id|order_date|order_status|order_value|item_id|quantity|item_price|product_name|   category| price|  cost| product_description|
+----------+--------+-----------+----------+------------+-----------+-------+--------+----------+------------+-----------+------+------+--------------------+
|     40914|       7|     484757|2026-02-20|   CANCELLED|    1960.62|     19|       3|    420.47|     despite|       Home|457.37|181.89|City others truth...|
|      4575|       7|     484757|2026-02-20|   CANCELLED|    1960.62|     20|       2|    434.22|         sea|Electronics|479.67| 88.54|Million manager h...|
|     22345|       7|     484757|2026-02-20|   CANCELLED|    1960.62|     21|       1|    374.77|       write|       Home|427.97|165.49|Alone computer th...|
+----------+--------+-----------+----------+--------

**<mark>We now add payment information</mark>.**

**Join:**

<mark>**orders_items_products_df**</mark>
        **JOIN**
<mark>**payments**</mark>

In [41]:
payments_df = spark.table("bronze_payments")

orders_full_df = (
    orders_items_products_df
    .join(
        payments_df,
        on="order_id",
        how="left"
    )
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 43, Finished, Available, Finished, False)

<mark>**Validate Join**</mark>

In [42]:
orders_full_df.show(3)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 44, Finished, Available, Finished, False)

+--------+----------+-----------+----------+------------+-----------+-------+--------+----------+------------+-----------+------+------+--------------------+----------+--------------+--------------+--------------+
|order_id|product_id|customer_id|order_date|order_status|order_value|item_id|quantity|item_price|product_name|   category| price|  cost| product_description|payment_id|payment_method|payment_status|payment_amount|
+--------+----------+-----------+----------+------------+-----------+-------+--------+----------+------------+-----------+------+------+--------------------+----------+--------------+--------------+--------------+
|       7|     40914|     484757|2026-02-20|   CANCELLED|    1960.62|     19|       3|    420.47|     despite|       Home|457.37|181.89|City others truth...|         7|           UPI|        FAILED|        421.13|
|       7|      4575|     484757|2026-02-20|   CANCELLED|    1960.62|     20|       2|    434.22|         sea|Electronics|479.67| 88.54|Million 

###### **<mark>Step 1 — Create Line Revenue</mark>**

**<mark>Business logic:</mark>**
**<mark>Revenue = quantity × item_price</mark>**

###### **<mark>Why This Step Matters</mark>**

**<mark>In real systems order_value is unreliable because: 
discounts, 
returns, 
tax adjustments 
 multiple items</mark>**

In [43]:
from pyspark.sql.functions import col

orders_full_df = orders_full_df.withColumn(
    "line_revenue",
    col("quantity") * col("item_price")
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 45, Finished, Available, Finished, False)

**<mark>Validation</mark>**

In [44]:
orders_full_df.select(
    "order_id",
    "product_id",
    "quantity",
    "item_price",
    "line_revenue"
).show(5)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 46, Finished, Available, Finished, False)

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|item_price|line_revenue|
+--------+----------+--------+----------+------------+
|      19|     41616|       5|     22.78|       113.9|
|      19|     26565|       2|    476.71|      953.42|
|      19|     41668|       2|     39.57|       79.14|
|      26|     21120|       2|     73.27|      146.54|
|      26|     24789|       1|    353.81|      353.81|
+--------+----------+--------+----------+------------+
only showing top 5 rows



###### **<mark>Next Transformation — Customer Lifetime Revenue (Window Function)</mark>**
**How much total revenue has each customer generated?**
**<mark>Customer Lifetime Value (CLV)</mark>**

In [45]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

customer_window = Window.partitionBy("customer_id")

orders_full_df = orders_full_df.withColumn(
    "customer_lifetime_revenue",
    sum("line_revenue").over(customer_window)
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 47, Finished, Available, Finished, False)

<mark>**<mark></mark>Validation**</mark>

In [46]:
orders_full_df.select(
    "customer_id",
    "order_id",
    "line_revenue",
    "customer_lifetime_revenue"
).show(10)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 48, Finished, Available, Finished, False)

+-----------+--------+------------+-------------------------+
|customer_id|order_id|line_revenue|customer_lifetime_revenue|
+-----------+--------+------------+-------------------------+
|          7| 5239604|     1523.12|                 10302.53|
|          7| 5239604|       291.0|                 10302.53|
|          7| 5239604|      338.82|                 10302.53|
|          7|   10361|      1043.4|                 10302.53|
|          7|   10361|      367.91|                 10302.53|
|          7|   10361|      192.52|                 10302.53|
|          7| 3489530|      439.31|                 10302.53|
|          7| 3489530|      776.68|                 10302.53|
|          7| 3489530|     1065.75|                 10302.53|
|          7| 6400938|      497.77|                 10302.53|
+-----------+--------+------------+-------------------------+
only showing top 10 rows



**<mark>Next Step — Customer Order Ranking**

**This is another very common real-world metric**.</mark>

**<mark>Business question:</mark>**

- **Which order was a customer's first order?**

- **Which order was their latest order?**

In [47]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

order_window = Window.partitionBy("customer_id").orderBy("order_date")

orders_full_df = orders_full_df.withColumn(
    "customer_order_rank",
    row_number().over(order_window)
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 49, Finished, Available, Finished, False)

<mark>**Validation**</mark>

In [48]:
orders_full_df.select(
    "customer_id",
    "order_id",
    "order_date",
    "customer_order_rank"
).show(10)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 50, Finished, Available, Finished, False)

+-----------+--------+----------+-------------------+
|customer_id|order_id|order_date|customer_order_rank|
+-----------+--------+----------+-------------------+
|          7| 3489530|2024-04-17|                  1|
|          7| 3489530|2024-04-17|                  2|
|          7| 3489530|2024-04-17|                  3|
|          7|   10361|2024-07-06|                  4|
|          7|   10361|2024-07-06|                  5|
|          7|   10361|2024-07-06|                  6|
|          7| 5549600|2024-12-14|                  7|
|          7| 5549600|2024-12-14|                  8|
|          7| 5549600|2024-12-14|                  9|
|          7| 5239604|2025-06-21|                 10|
+-----------+--------+----------+-------------------+
only showing top 10 rows



**<mark>Next Step → Running Customer Revenue</mark>**

<mark>**Now we calculate cumulative revenue per customer over time**.</mark>

<mark>**Business question:Now we calculate cumulative revenue per customer over time**</mark>.



In [49]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

running_window = Window.partitionBy("customer_id").orderBy("order_date")

orders_full_df = orders_full_df.withColumn(
    "running_customer_revenue",
    sum("line_revenue").over(running_window)
)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 51, Finished, Available, Finished, False)

**<mark>Validation of Running Revenue calculation</mark>**

In [50]:
orders_full_df.select(
    "customer_id",
    "order_date",
    "line_revenue",
    "running_customer_revenue"
).show(10)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 52, Finished, Available, Finished, False)

+-----------+----------+------------+------------------------+
|customer_id|order_date|line_revenue|running_customer_revenue|
+-----------+----------+------------+------------------------+
|          7|2024-04-17|      439.31|                 2281.74|
|          7|2024-04-17|      776.68|                 2281.74|
|          7|2024-04-17|     1065.75|                 2281.74|
|          7|2024-07-06|      1043.4|      3885.5699999999997|
|          7|2024-07-06|      367.91|      3885.5699999999997|
|          7|2024-07-06|      192.52|      3885.5699999999997|
|          7|2024-12-14|      908.46|                 6453.93|
|          7|2024-12-14|      1401.9|                 6453.93|
|          7|2024-12-14|       258.0|                 6453.93|
|          7|2025-06-21|     1523.12|       8606.869999999999|
+-----------+----------+------------+------------------------+
only showing top 10 rows



**<mark>Write Final Silver Dataset</mark>**

In [51]:
orders_full_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders_enriched")

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 53, Finished, Available, Finished, False)

In [52]:
spark.sql("SELECT COUNT(*) FROM silver_orders_enriched").show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 54, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|24000000|
+--------+



spark.sql("""
DESCRIBE HISTORY silver_orders_enriched
""").show(truncate=False)

In [56]:
spark.sql("""
DESCRIBE HISTORY silver_orders_enriched
""").show(truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 58, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------------+------------+-------------------------------------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                         |userMetadata|engineInfo                                                   |
+-------+-----------------------+------+--------+-----------------

###### **<mark>Create a New Version (Time Travel -Update Simulation)</mark>**

In [58]:
spark.sql("""
UPDATE silver_orders_enriched
SET item_price = item_price * 1.10
WHERE order_status = 'DELIVERED'
""")

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 60, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]

**<mark>Verify History Again</mark>**

In [59]:
spark.sql("""
DESCRIBE HISTORY silver_orders_enriched
""").show(truncate=False)

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 61, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------------------------------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                     |job |n

###### **<mark>Query Previous Version (Time Travel)</mark>**
**Now we query Version 0.**

In [ ]:
spark.sql("""
SELECT *
FROM silver_orders_enriched
VERSION AS OF 0
LIMIT 3
""").show()

###### **<mark>Query Latest Version</mark>**

In [62]:
spark.sql("""
SELECT *
FROM silver_orders_enriched
VERSION AS OF 1
LIMIT 3
""").show()

StatementMeta(, c97810b5-8fcb-40f7-b964-a6eda1aab7c2, 64, Finished, Available, Finished, False)

+--------+----------+-----------+----------+------------+-----------+--------+--------+----------+------------+--------+------+-----+--------------------+----------+--------------+--------------+--------------+------------+-------------------------+-------------------+------------------------+
|order_id|product_id|customer_id|order_date|order_status|order_value| item_id|quantity|item_price|product_name|category| price| cost| product_description|payment_id|payment_method|payment_status|payment_amount|line_revenue|customer_lifetime_revenue|customer_order_rank|running_customer_revenue|
+--------+----------+-----------+----------+------------+-----------+--------+--------+----------+------------+--------+------+-----+--------------------+----------+--------------+--------------+--------------+------------+-------------------------+-------------------+------------------------+
| 7649063|     23435|         11|2024-05-20|     SHIPPED|     105.67|22947187|       5|    224.47|         and|  Be